# Validation Verdict Analysis

This notebook visualizes verdict statistics from the trace annotation validator app.
It is scoped to `validation/misleading` records stored in the PromptResultDB.

Tags like `llm_provider:*`, `llm_provider_model:*`, `verdict:*` are expected to be
present on all validator-created records. Problem tags (`difficulty:*`, `source:*`)
are dataset-dependent and may not always be present.

In [ ]:
import datetime

import matplotlib.pyplot as plt
import pandas as pd

import pyine.prompts.result_db
import pyine.utils.reprod

pyine.utils.reprod.entrypoint_setup()

In [ ]:
# --- EDIT THESE SETTINGS IF NEEDED ---

VALIDATION_PROMPT_NAME = "validation/misleading"
TAG_FILTER_RULE: str | None = None  # e.g. "+llm_provider:openai" to keep only openai
MAX_RESULT_AGE: datetime.timedelta | None = None  # e.g. datetime.timedelta(days=7)

# path to the PromptResultDB; set to None to use the framework default, or provide
# the path used when running the trace annotation validator (--db-path argument)
DB_PATH: str | None = None

In [ ]:
db = pyine.prompts.result_db.PromptResultDB(DB_PATH) if DB_PATH else pyine.prompts.result_db.get_framework_db()
print(f"Using PromptResultDB at: {db._path} ({db.count_entries()} total records)")

available_prompt_names = db.list_prompt_names()
if VALIDATION_PROMPT_NAME not in available_prompt_names:
    print(f"\nWARNING: prompt name '{VALIDATION_PROMPT_NAME}' not found in this DB.")
    print(f"Available prompt names ({len(available_prompt_names)}):")
    for pn in available_prompt_names:
        print(f"  - {pn}")
    print("\nIf the validator was run with --db-path, set DB_PATH in the settings cell above.")

summaries = db.get_summaries_by_prompt_name(
    VALIDATION_PROMPT_NAME,
    tag_filter_rule=TAG_FILTER_RULE,
    max_result_age=MAX_RESULT_AGE,
)
print(f"\nLoaded {len(summaries)} validation summaries")
if not summaries:
    raise ValueError("No records found; check VALIDATION_PROMPT_NAME and DB_PATH in the settings above.")

In [ ]:
def _extract_tag_value(tags: list[str], prefix: str, default: str = "unknown") -> str:
    """Extract the value of the first tag matching a prefix like 'llm_provider:'."""
    for tag in tags:
        if tag.startswith(prefix):
            return tag[len(prefix) :]
    return default


rows = []
for summ in summaries:
    verdict = summ.meta.get("verdict", "UNKNOWN") if summ.meta else "UNKNOWN"
    rows.append(
        {
            "identifier": summ.identifier,
            "verdict": str(verdict),
            "explanation": str(summ.meta.get("explanation", "")) if summ.meta else "",
            "source_prompt_name": str(summ.meta.get("source_prompt_name", "")) if summ.meta else "",
            "source_record_uid": str(summ.meta.get("source_record_uid", "")) if summ.meta else "",
            "source_identifier": str(summ.meta.get("source_identifier", "")) if summ.meta else "",
            "llm_provider": _extract_tag_value(summ.tags, "llm_provider:"),
            "llm_model": _extract_tag_value(summ.tags, "llm_provider_model:"),
            "difficulty": _extract_tag_value(summ.tags, "difficulty:"),
            "source": _extract_tag_value(summ.tags, "source:"),
            "created_at": summ.created_at,
            "tags": summ.tags,
        }
    )

df = pd.DataFrame(rows)
df["date"] = df["created_at"].dt.date
print(f"DataFrame shape: {df.shape}")
df.head()

In [ ]:
# --- verdict distribution ---
verdict_colors = {
    "MISLEADING": "#d32f2f",
    "NOT_MISLEADING": "#388e3c",
    "UNINFORMATIVE": "#f57c00",
}
verdict_counts = df["verdict"].value_counts()
verdict_pcts = df["verdict"].value_counts(normalize=True) * 100
summary_table = pd.DataFrame({"count": verdict_counts, "percent": verdict_pcts.round(1)})
print(summary_table)

fig, ax = plt.subplots(figsize=(6, 4))
colors = [verdict_colors.get(v, "#9e9e9e") for v in verdict_counts.index]
verdict_counts.plot.bar(ax=ax, color=colors, edgecolor="black", linewidth=0.5)
ax.set_title("Verdict Distribution")
ax.set_ylabel("Count")
ax.set_xlabel("Verdict")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# --- breakdown by source prompt name ---
if df["source_prompt_name"].nunique() > 1 or (df["source_prompt_name"] != "").any():
    ct_source = pd.crosstab(df["source_prompt_name"], df["verdict"])
    print(ct_source)
    fig, ax = plt.subplots(figsize=(max(6, ct_source.shape[0] * 1.2), 5))
    ordered_cols = [c for c in ["MISLEADING", "NOT_MISLEADING", "UNINFORMATIVE"] if c in ct_source.columns]
    bar_colors = [verdict_colors.get(c, "#9e9e9e") for c in ordered_cols]
    ct_source[ordered_cols].plot.bar(stacked=True, ax=ax, color=bar_colors, edgecolor="black", linewidth=0.5)
    ax.set_title("Verdicts by Source Prompt Name")
    ax.set_ylabel("Count")
    ax.set_xlabel("Source Prompt Name")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("No source_prompt_name data available (single or empty value); skipping.")

In [ ]:
# --- breakdown by LLM model ---
if df["llm_model"].nunique() > 0 and (df["llm_model"] != "unknown").any():
    ct_model = pd.crosstab(df["llm_model"], df["verdict"])
    print(ct_model)
    # misleading rate per model
    if "MISLEADING" in ct_model.columns:
        misleading_rate = (ct_model["MISLEADING"] / ct_model.sum(axis=1) * 100).round(1)
        print("\nMisleading rate per model (%):\n", misleading_rate.to_string())
    fig, ax = plt.subplots(figsize=(max(6, ct_model.shape[0] * 1.2), 5))
    ordered_cols = [c for c in ["MISLEADING", "NOT_MISLEADING", "UNINFORMATIVE"] if c in ct_model.columns]
    bar_colors = [verdict_colors.get(c, "#9e9e9e") for c in ordered_cols]
    ct_model[ordered_cols].plot.bar(stacked=True, ax=ax, color=bar_colors, edgecolor="black", linewidth=0.5)
    ax.set_title("Verdicts by LLM Model")
    ax.set_ylabel("Count")
    ax.set_xlabel("LLM Model")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("No llm_model data available; skipping.")

In [ ]:
# --- breakdown by difficulty ---
if df["difficulty"].nunique() > 1:
    ct_diff = pd.crosstab(df["difficulty"], df["verdict"])
    print(ct_diff)
    fig, ax = plt.subplots(figsize=(max(6, ct_diff.shape[0] * 1.2), 5))
    ordered_cols = [c for c in ["MISLEADING", "NOT_MISLEADING", "UNINFORMATIVE"] if c in ct_diff.columns]
    bar_colors = [verdict_colors.get(c, "#9e9e9e") for c in ordered_cols]
    ct_diff[ordered_cols].plot.bar(stacked=True, ax=ax, color=bar_colors, edgecolor="black", linewidth=0.5)
    ax.set_title("Verdicts by Difficulty")
    ax.set_ylabel("Count")
    ax.set_xlabel("Difficulty")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print(f"Only one difficulty value ({df['difficulty'].unique()[0]}); skipping chart.")

In [ ]:
# --- summary stats ---
total = len(df)
verdict_counts_dict = df["verdict"].value_counts().to_dict()
non_uninformative = total - verdict_counts_dict.get("UNINFORMATIVE", 0)
misleading_count = verdict_counts_dict.get("MISLEADING", 0)
misleading_rate = (misleading_count / non_uninformative * 100) if non_uninformative > 0 else 0.0
unique_identifiers = df["identifier"].nunique()
date_range = f"{df['created_at'].min():%Y-%m-%d %H:%M} to {df['created_at'].max():%Y-%m-%d %H:%M} UTC"

print("=== Summary Statistics ===")
print(f"Total records: {total}")
for verdict_val, count in sorted(verdict_counts_dict.items()):
    print(f"  {verdict_val}: {count}")
print(f"Misleading rate (excl. UNINFORMATIVE): {misleading_rate:.1f}%")
print(f"Unique identifiers: {unique_identifiers}")
print(f"Date range: {date_range}")